In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import argparse
from copy import deepcopy

from layers import disp_to_depth, SSIM, BackprojectDepth, Project3D, transformation_from_parameters, get_smooth_loss
from utils import readlines, normalize_image
import networks

In [ ]:
# 1. 组合 encoder+decoder 为一个整体模型，只返回主Tensor
class SQLDepthModel(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    def forward(self, x):
        features = self.encoder(x)
        out = self.decoder(features)
        return out[("disp", 0)]  # 只返回主Tensor

In [ ]:
# 2. 自监督TTA适应模块
class SelfSupervisedTTA(nn.Module):
    def __init__(self, depth_model, pose_model, depth_optimizer, steps=1, 
                 photo_weight=1.0, smooth_weight=0.1, min_depth=0.1, max_depth=100.0, 
                 batch_size=4, height=192, width=640):
        super().__init__()
        self.depth_model = depth_model
        self.pose_model = pose_model
        self.depth_optimizer = depth_optimizer
        self.steps = steps
        
        # 损失权重
        self.photo_weight = photo_weight
        self.smooth_weight = smooth_weight
        
        # 深度范围
        self.min_depth = min_depth
        self.max_depth = max_depth
        
        # 创建SSIM损失
        self.ssim = SSIM()
        self.ssim.to(next(depth_model.parameters()).device)
        
        # 创建投影工具
        self.backproject_depth = {}
        self.project_3d = {}
        
        # 只需要一个尺度
        scale = 0
        h = height
        w = width
        
        self.backproject_depth[scale] = BackprojectDepth(batch_size, h, w)
        self.backproject_depth[scale].to(next(depth_model.parameters()).device)
        
        self.project_3d[scale] = Project3D(batch_size, h, w)
        self.project_3d[scale].to(next(depth_model.parameters()).device)
        
    def compute_reprojection_loss(self, pred, target):
        """使用与原始自监督深度估计相同的重投影损失"""
        abs_diff = torch.abs(target - pred)
        l1_loss = abs_diff.mean(1, True)
        
        ssim_loss = self.ssim(pred, target).mean(1, True)
        reprojection_loss = 0.85 * ssim_loss + 0.15 * l1_loss
        
        return reprojection_loss
    
    def forward(self, target_frame, source_frames, K=None, inv_K=None, clean_depth=None):
        """执行自监督测试时适应
        Args:
            target_frame: 目标帧 [B, 3, H, W]
            source_frames: 源帧列表 [[B, 3, H, W], ...]
            K: 相机内参 [B, 4, 4]
            inv_K: 相机内参逆矩阵 [B, 4, 4]
            clean_depth: 可选的伪标签视差 [B, H, W]
        """
        self.depth_model.train()
        # 位姿网络保持评估模式
        self.pose_model[0].eval()
        self.pose_model[1].eval()
        
        device = target_frame.device
        batch_size, _, height, width = target_frame.shape
        
        # 如果没有提供内参，使用默认值
        if K is None:
            K = torch.zeros(batch_size, 4, 4, device=device)
            K[:, 0, 0] = 0.58 * width  # fx
            K[:, 1, 1] = 0.58 * height  # fy
            K[:, 0, 2] = 0.5 * width  # cx
            K[:, 1, 2] = 0.5 * height  # cy
            K[:, 2, 2] = 1
            K[:, 3, 3] = 1
        
        if inv_K is None:
            inv_K = torch.inverse(K)
        
        for _ in range(self.steps):
            total_loss = 0
            
            # 构建输入和输出字典
            inputs = {}
            outputs = {}
            
            # 添加帧和相机参数到输入
            inputs[("color", 0, 0)] = target_frame
            inputs[("K", 0)] = K
            inputs[("inv_K", 0)] = inv_K
            
            # 帧ID (0是目标帧，-1和1是源帧)
            frame_ids = [0]
            for i, _ in enumerate(source_frames):
                frame_id = -1 if i == 0 else 1  # 假设最多两个源帧
                inputs[("color", frame_id, 0)] = source_frames[i]
                frame_ids.append(frame_id)
            
            # 预测目标帧深度
            features = self.depth_model.encoder(target_frame)
            depth_outputs = self.depth_model.decoder(features)
            
            # 深度预测
            outputs[("disp", 0)] = depth_outputs[("disp", 0)]
            disp = outputs[("disp", 0)]
            
            # 伪标签损失（如果提供）
            if clean_depth is not None:
                if clean_depth.dim() == 3:
                    clean_depth = clean_depth.unsqueeze(1)
                clean_loss = F.l1_loss(disp, clean_depth)
                total_loss += clean_loss
            
            # 转换视差为深度
            _, depth = disp_to_depth(disp, self.min_depth, self.max_depth)
            outputs[("depth", 0, 0)] = depth
            
            # 预测位姿 (不进行梯度计算)
            with torch.no_grad():
                for i, frame_id in enumerate(frame_ids[1:]):
                    # 准备位姿网络输入
                    pose_inputs = torch.cat([target_frame, inputs[("color", frame_id, 0)]], 1)
                    
                    # 预测位姿
                    pose_outputs = self.pose_model[1]([self.pose_model[0](pose_inputs)])
                    axisangle, translation = pose_outputs
                    
                    outputs[("axisangle", 0, frame_id)] = axisangle
                    outputs[("translation", 0, frame_id)] = translation
                    
                    # 转换为变换矩阵
                    outputs[("cam_T_cam", 0, frame_id)] = transformation_from_parameters(
                        axisangle[:, 0], translation[:, 0], invert=(frame_id < 0))
            
            # 生成重投影图像
            scale = 0  # 只使用一个尺度
            
            for frame_id in frame_ids[1:]:
                T = outputs[("cam_T_cam", 0, frame_id)]
                
                # 反投影到3D并投影到其他视角
                cam_points = self.backproject_depth[scale](depth, inputs[("inv_K", scale)])
                pix_coords = self.project_3d[scale](cam_points, inputs[("K", scale)], T)
                
                outputs[("sample", frame_id, scale)] = pix_coords
                
                # 对源帧进行采样以获得重投影图像
                outputs[("color", frame_id, scale)] = F.grid_sample(
                    inputs[("color", frame_id, scale)],
                    outputs[("sample", frame_id, scale)],
                    padding_mode="border", align_corners=True)
                
                # 自动掩码的身份映射图像
                outputs[("color_identity", frame_id, scale)] = inputs[("color", frame_id, scale)]
            
            # 计算损失
            # 1. 重投影损失
            reprojection_losses = []
            
            for frame_id in frame_ids[1:]:
                pred = outputs[("color", frame_id, scale)]
                target = inputs[("color", 0, scale)]
                reprojection_losses.append(self.compute_reprojection_loss(pred, target))
            
            reprojection_losses = torch.cat(reprojection_losses, 1)
            
            # 身份重投影损失（自动掩码）
            identity_reprojection_losses = []
            for frame_id in frame_ids[1:]:
                pred = inputs[("color", frame_id, scale)]
                target = inputs[("color", 0, scale)]
                identity_reprojection_losses.append(self.compute_reprojection_loss(pred, target))
            
            identity_reprojection_losses = torch.cat(identity_reprojection_losses, 1)
            
            # 添加随机数以打破平局
            identity_reprojection_losses += torch.randn(
                identity_reprojection_losses.shape, device=device) * 0.00001
            
            # 组合损失并取最小值
            combined = torch.cat((identity_reprojection_losses, reprojection_losses), dim=1)
            to_optimise, idxs = torch.min(combined, dim=1)
            
            # 自动掩码
            outputs["identity_selection/{}".format(scale)] = (
                idxs > identity_reprojection_losses.shape[1] - 1).float()
            
            # 2. 平滑损失
            mean_disp = disp.mean(2, True).mean(3, True)
            norm_disp = disp / (mean_disp + 1e-7)
            smooth_loss = get_smooth_loss(norm_disp, target_frame)
            
            # 组合所有损失
            photo_loss = to_optimise.mean()
            smooth_loss = smooth_loss
            
            loss = self.photo_weight * photo_loss + self.smooth_weight * smooth_loss
            total_loss += loss
            
            # 反向传播 (只更新深度模型)
            self.depth_optimizer.zero_grad()
            total_loss.backward()
            self.depth_optimizer.step()
        
        # 切换回评估模式并返回深度预测
        self.depth_model.eval()
        with torch.no_grad():
            features = self.depth_model.encoder(target_frame)
            out = self.depth_model.decoder(features)
            return out[("disp", 0)]

In [ ]:
# 三种更新模式配置函数
def configure_model_for_tta(depth_model, update_mode='full'):
    """配置模型进行测试时适应
    
    Args:
        depth_model: 深度模型
        update_mode: 更新模式，可选 'bn_only', 'bn_decoder', 'full'
    """
    # 首先冻结所有参数
    for param in depth_model.parameters():
        param.requires_grad = False
        
    if update_mode == 'bn_only':
        # 只解冻BN层
        for m in depth_model.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.requires_grad_(True)
                m.track_running_stats = False
                m.running_mean = None
                m.running_var = None
                for p in m.parameters():
                    p.requires_grad = True
                    
    elif update_mode == 'bn_decoder':
        # 解冻BN层和Decoder
        for m in depth_model.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.requires_grad_(True)
                m.track_running_stats = False
                m.running_mean = None
                m.running_var = None
                for p in m.parameters():
                    p.requires_grad = True
        
        # 解冻decoder
        for param in depth_model.decoder.parameters():
            param.requires_grad = True
            
    elif update_mode == 'full':
        # 解冻所有参数
        for param in depth_model.parameters():
            param.requires_grad = True
            
        # 特殊处理BN层
        for m in depth_model.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.track_running_stats = False
                m.running_mean = None
                m.running_var = None
    
    return depth_model


# 收集需要更新的参数
def collect_tta_params(depth_model):
    """收集TTA需要更新的参数"""
    params = []
    for name, param in depth_model.named_parameters():
        if param.requires_grad:
            params.append(param)
    return params

'''
# 3. 原始EATA模块保持不变
class EATA(nn.Module):
    def __init__(self, model, optimizer, steps=1):
        super().__init__()
        self.model = model
        self.optimizer = optimizer
        self.steps = steps
        self.criterion = nn.L1Loss()
    def forward(self, x, clean_depth=None):
        self.model.train()
        for _ in range(self.steps):
            out = self.model(x)
            if clean_depth is not None:
                if clean_depth.dim() == 3:
                    clean_depth = clean_depth.unsqueeze(1)  # [B, 1, H, W]
                # 假设out和clean_depth shape一致
                loss = self.criterion(out, clean_depth)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
        self.model.eval()
        with torch.no_grad():
            return self.model(x)
'''

def compute_errors(gt, pred):
    thresh = np.maximum((gt / pred), (pred / gt))
    a1 = (thresh < 1.25     ).mean()
    a2 = (thresh < 1.25 ** 2).mean()
    a3 = (thresh < 1.25 ** 3).mean()
    rmse = (gt - pred) ** 2
    rmse = np.sqrt(rmse.mean())
    rmse_log = (np.log(gt) - np.log(pred)) ** 2
    rmse_log = np.sqrt(rmse_log.mean())
    abs_rel = np.mean(np.abs(gt - pred) / gt)
    sq_rel = np.mean(((gt - pred) ** 2) / gt)
    return abs_rel, sq_rel, rmse, rmse_log, a1, a2, a3

In [ ]:
def main():
    parser = argparse.ArgumentParser(fromfile_prefix_chars='@')
    # 原有参数
    parser.add_argument('--mode', type=str, choices=['generate_clean', 'adapt'], required=True, help='generate_clean:生成clean伪标签; adapt:TTA自适应')
    parser.add_argument('--data_path', type=str, required=True)
    parser.add_argument('--log_dir', type=str, required=True)
    parser.add_argument('--model_name', type=str, required=True)
    parser.add_argument('--dataset', type=str, default='kitti')
    parser.add_argument('--eval_split', type=str, default='eigen')
    parser.add_argument('--backbone', type=str, default='resnet_lite')
    parser.add_argument('--height', type=int, default=192)
    parser.add_argument('--width', type=int, default=640)
    parser.add_argument('--batch_size', type=int, default=16)
    parser.add_argument('--num_layers', type=int, default=50)
    parser.add_argument('--num_features', type=int, default=256)
    parser.add_argument('--model_dim', type=int, default=32)
    parser.add_argument('--patch_size', type=int, default=16)
    parser.add_argument('--dim_out', type=int, default=64)
    parser.add_argument('--query_nums', type=int, default=64)
    parser.add_argument('--min_depth', type=float, default=0.001)
    parser.add_argument('--max_depth', type=float, default=80.0)
    parser.add_argument('--load_weights_folder', type=str, required=True)
    parser.add_argument('--post_process', action='store_true')
    parser.add_argument('--disable_median_scaling', action='store_true')
    parser.add_argument('--pred_depth_scale_factor', type=float, default=1.0)
    parser.add_argument('--eata_steps', type=int, default=1)
    parser.add_argument('--clean_pred_path', type=str, default='clean_pred_disps.npy', help='clean伪标签保存/加载路径')
    
    # 添加TTA参数
    # parser.add_argument('--use_ss_tta', action='store_true', help='使用自监督TTA而非EATA')
    parser.add_argument('--pose_weights_folder', type=str, default=None, help='位姿网络权重文件夹')
    parser.add_argument('--photo_weight', type=float, default=1.0, help='光度损失权重')
    parser.add_argument('--smooth_weight', type=float, default=0.1, help='平滑损失权重')
    parser.add_argument('--update_mode', type=str, default='full', 
                      choices=['bn_only', 'bn_decoder', 'full'], 
                      help='参数更新模式: bn_only=只更新BN层, bn_decoder=更新BN层+Decoder, full=更新全网络')
    
    args = parser.parse_args()

    # === 数据集 ===
    splits_dir = os.path.join(os.path.dirname(__file__), "splits")
    filenames = readlines(os.path.join(splits_dir, args.eval_split, "test_files.txt"))
    
    # 如果使用自监督TTA，需要相邻帧
    from datasets.kitti_dataset import KITTIRAWDataset

    frame_ids = [0, -1, 1]  # 加载当前帧和相邻帧
        
    dataset = KITTIRAWDataset(
        args.data_path, filenames,
        args.height, args.width,
        frame_ids, 4, is_train=False
    )
    
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=args.batch_size, shuffle=False, num_workers=4, pin_memory=True, drop_last=False)

    # === 加载深度网络 ===
    if args.backbone in ["resnet", "resnet_lite"]:
        encoder = networks.ResnetEncoderDecoder(
            num_layers=args.num_layers,
            num_features=args.num_features,
            model_dim=args.model_dim
        )
    elif args.backbone == "resnet18_lite":
        encoder = networks.LiteResnetEncoderDecoder(model_dim=args.model_dim)
    elif args.backbone == "eff_b5":
        encoder = networks.BaseEncoder.build(num_features=args.num_features, model_dim=args.model_dim)
    else:
        encoder = networks.Unet(pretrained=False, backbone=args.backbone, in_channels=3, num_classes=args.model_dim, decoder_channels=[1024, 512, 256, 128])

    encoder_path = os.path.join(args.load_weights_folder, "encoder.pth")
    encoder_dict = torch.load(encoder_path)
    model_dict = encoder.state_dict()
    encoder.load_state_dict({k: v for k, v in encoder_dict.items() if k in model_dict})
    encoder.cuda().eval()

    if args.backbone.endswith("_lite"):
        decoder = networks.Lite_Depth_Decoder_QueryTr(
            in_channels=args.model_dim,
            patch_size=args.patch_size,
            dim_out=args.dim_out,
            embedding_dim=args.model_dim,
            query_nums=args.query_nums,
            num_heads=4,
            min_val=args.min_depth,
            max_val=args.max_depth
        )
    else:
        decoder = networks.Depth_Decoder_QueryTr(
            in_channels=args.model_dim,
            patch_size=args.patch_size,
            dim_out=args.dim_out,
            embedding_dim=args.model_dim,
            query_nums=args.query_nums,
            num_heads=4,
            min_val=args.min_depth,
            max_val=args.max_depth
        )
    decoder_path = os.path.join(args.load_weights_folder, "depth.pth")
    decoder.load_state_dict(torch.load(decoder_path))
    decoder.cuda().eval()

    depth_model = SQLDepthModel(encoder, decoder)

    # 配置模型进行TTA (根据更新模式)
    depth_model = configure_model_for_tta(depth_model, args.update_mode)
    
    # 收集需要更新的参数并创建优化器
    params_to_update = collect_tta_params(depth_model)
    depth_optimizer = torch.optim.Adam(params_to_update, lr=4e-5)

    # === 如果使用自监督TTA，加载位姿网络 ===
    # 位姿网络
    pose_encoder = networks.ResnetEncoder(18, True, num_input_images=2)
    pose_encoder.cuda()
        
    pose_decoder = networks.PoseDecoder(
        pose_encoder.num_ch_enc,
        num_input_features=1,
        num_frames_to_predict_for=2
    )
    pose_decoder.cuda()
        
    # 加载位姿网络权重（如果提供）
    if args.pose_weights_folder is not None:
        pose_encoder_path = os.path.join(args.pose_weights_folder, "pose_encoder.pth")
        pose_decoder_path = os.path.join(args.pose_weights_folder, "pose.pth")
        if os.path.exists(pose_encoder_path) and os.path.exists(pose_decoder_path):
            pose_encoder.load_state_dict(torch.load(pose_encoder_path))
            pose_decoder.load_state_dict(torch.load(pose_decoder_path))
            print("Loaded pose network weights")
        
    # 位姿网络设为评估模式并冻结参数
    pose_encoder.eval()
    pose_decoder.eval()
    for param in pose_encoder.parameters():
        param.requires_grad = False
    for param in pose_decoder.parameters():
        param.requires_grad = False
        
    # 创建自监督TTA模型
    adapt_model = SelfSupervisedTTA(
        depth_model, [pose_encoder, pose_decoder],
        depth_optimizer,
        steps=args.eata_steps,
        photo_weight=args.photo_weight,
        smooth_weight=args.smooth_weight,
        min_depth=args.min_depth,
        max_depth=args.max_depth,
        batch_size=args.batch_size,
        height=args.height,
        width=args.width
    )

    if args.mode == 'generate_clean':
        # === 生成clean伪标签 ===
        depth_model.eval()
        pred_disps = []
        with torch.no_grad():
            for batch in dataloader:
                images = batch[("color", 0, 0)].cuda()
                pred_disp = depth_model(images)
                pred_disp = pred_disp.cpu().numpy()
                if pred_disp.ndim == 4:
                    pred_disp = pred_disp[:, 0, :, :]
                pred_disps.append(pred_disp)
        pred_disps = np.concatenate(pred_disps, axis=0)
        np.save(args.clean_pred_path, pred_disps)
        print(f"Clean伪标签已保存到 {args.clean_pred_path}")
        return

    elif args.mode == 'adapt':
        # === TTA自适应推理 ===
        clean_pred_disps = np.load(args.clean_pred_path)
        
        pred_disps = []
        idx = 0
        
        for batch_idx, batch in enumerate(dataloader):
            images = batch[("color", 0, 0)].cuda()
            # 取对应clean伪标签
            batch_size = images.shape[0]
            clean_depth = torch.from_numpy(clean_pred_disps[idx:idx+batch_size]).cuda()
            idx += batch_size
            
            # 自监督TTA需要相邻帧和内参
            source_frames = []
            for i in [-1, 1]:
                if ("color", i, 0) in batch:
                    source_frames.append(batch[("color", i, 0)].cuda())
                
            # 获取相机内参
            K = None
            inv_K = None
            if ("K", 0) in batch and ("inv_K", 0) in batch:
                K = batch[("K", 0)].cuda()
                inv_K = batch[("inv_K", 0)].cuda()
                
            pred_disp = adapt_model(images, source_frames, K, inv_K, clean_depth)
                
            pred_disp = pred_disp.cpu().numpy()
            if pred_disp.ndim == 4:
                pred_disp = pred_disp[:, 0, :, :]
            pred_disps.append(pred_disp)
            
            if batch_idx % 10 == 0:
                print(f"处理批次 {batch_idx}/{len(dataloader)}")
            
        pred_disps = np.concatenate(pred_disps, axis=0)

        # === 评测 ===
        gt_path = os.path.join(splits_dir, args.eval_split, "gt_depths.npz")
        gt_depths = np.load(gt_path, fix_imports=True, encoding='latin1', allow_pickle=True)["data"]
        errors = []
        ratios = []
        MIN_DEPTH = args.min_depth
        MAX_DEPTH = args.max_depth

        for i in range(pred_disps.shape[0]):
            gt_depth = gt_depths[i]
            gt_height, gt_width = gt_depth.shape[:2]
            pred_disp = pred_disps[i]
            pred_disp = cv2.resize(pred_disp, (gt_width, gt_height))
            pred_depth = pred_disp

            mask = np.logical_and(gt_depth > MIN_DEPTH, gt_depth < MAX_DEPTH)
            crop = np.array([0.40810811 * gt_height, 0.99189189 * gt_height,
                             0.03594771 * gt_width,  0.96405229 * gt_width]).astype(np.int32)
            crop_mask = np.zeros(mask.shape)
            crop_mask[crop[0]:crop[1], crop[2]:crop[3]] = 1
            mask = np.logical_and(mask, crop_mask)

            pred_depth = pred_depth[mask]
            gt_depth = gt_depth[mask]

            pred_depth *= args.pred_depth_scale_factor
            if not args.disable_median_scaling:
                ratio = np.median(gt_depth) / np.median(pred_depth)
                ratios.append(ratio)
                pred_depth *= ratio

            pred_depth[pred_depth < MIN_DEPTH] = MIN_DEPTH
            pred_depth[pred_depth > MAX_DEPTH] = MAX_DEPTH

            errors.append(compute_errors(gt_depth, pred_depth))

        if not args.disable_median_scaling:
            ratios = np.array(ratios)
            med = np.median(ratios)
            print(" Scaling ratios | med: {:0.3f} | std: {:0.3f}".format(med, np.std(ratios / med)))

        mean_errors = np.array(errors).mean(0)
        print("\n  " + ("{:>8} | " * 7).format("abs_rel", "sq_rel", "rmse", "rmse_log", "a1", "a2", "a3"))
        print(("&{: 8.3f}  " * 7).format(*mean_errors.tolist()) + "\\\\")
        print("\n-> Done!")

In [ ]:
if __name__ == "__main__":
    main()